# Data Preparation

In [2]:
# Importing modules
from utils import seed_everything, train_and_evaluate, test
from data_processing import load_pairs, label_pairs
from data_siamese import get_train_transform, get_val_transform, get_test_transform, SiameseDataset
from model import SiameseNetwork
from loss import ContrastiveLoss
import torch
from torch.utils.data import DataLoader
import torch.optim as optim
import os
import json
import numpy as np

# Configurations
SEED = 42
DIR = os.path.abspath("./data/sign/")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(SEED)

In [3]:
# Load different, same pairs and label them
diff_train_pairs, same_train_pairs, diff_val_pairs, same_val_pairs, diff_test_pairs, same_test_pairs = load_pairs(DIR)
train_images_1, train_images_2, train_labels = label_pairs(diff_train_pairs, same_train_pairs)
val_images_1, val_images_2, val_labels = label_pairs(diff_val_pairs, same_val_pairs)
test_images_1, test_images_2, test_labels = label_pairs(diff_test_pairs, same_test_pairs)

# Create Siamese datasets from image pairs, labels, and transformations
train_transform = get_train_transform()
val_transform = get_val_transform()
test_transform = get_test_transform()
train_dataset = SiameseDataset(train_images_1, train_images_2, train_labels, transform=train_transform)
val_dataset = SiameseDataset(val_images_1, val_images_2, val_labels, transform=val_transform)
test_dataset = SiameseDataset(test_images_1, test_images_2, test_labels, test_transform)

# Create DataLoader for datasets
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)

# Train and Validate

In [14]:
# Create Siamese model with different backbones, loss function, optimizer, and scheduler
backbone = 'resnet18' # 'efficientnet_b0', 'efficientnet_b2', 'resnet18', 'resnet50'
model = SiameseNetwork(model_name=backbone, pretrained=True)
model.to(device)
criterion = ContrastiveLoss(margin=1.0)
optimizer = optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-6)

# Train and evaluate model, save performance statistics and weights of the best one
save_weights_path = f'weights/best_{backbone}_model.pth'
os.makedirs(os.path.dirname(save_weights_path), exist_ok=True)

best_threshold, all_epoch_scores, best_epoch_stats = train_and_evaluate(model, train_loader, val_loader, num_epochs=20, criterion=criterion, optimizer=optimizer, scheduler=scheduler, device=device, save_path=save_weights_path)

performance_data = {
    "all_epoch_scores": all_epoch_scores,
    "best_epoch_stats": best_epoch_stats
}
save_stats_path = f'stats/val/performance_{backbone}_stats.json'
os.makedirs(os.path.dirname(save_stats_path), exist_ok=True)

with open(save_stats_path, "w") as f:
    json.dump(performance_data, f, indent=4)
print("Training complete. Stats saved.")


100%|██████████| 211/211 [00:55<00:00,  3.83it/s]


Epoch [1/20] | Train Loss: 0.2722, Val Loss: 0.3339 | Tuned Thresold: 0.4938 | Val Accuracy: 0.5541, Precision: 0.5346, Recall: 0.9415, F1: 0.6819
Saved the best model with F1: 0.6819


100%|██████████| 211/211 [00:55<00:00,  3.83it/s]


Epoch [2/20] | Train Loss: 0.2684, Val Loss: 0.4162 | Tuned Thresold: 0.2161 | Val Accuracy: 0.5719, Precision: 0.5472, Recall: 0.9087, F1: 0.6831
Saved the best model with F1: 0.6831


100%|██████████| 211/211 [00:56<00:00,  3.75it/s]


...

Epoch [18/20] | Train Loss: 0.2507, Val Loss: 0.3041 | Tuned Thresold: 0.7330 | Val Accuracy: 0.5256, Precision: 0.5174, Recall: 0.9742, F1: 0.6759


100%|██████████| 211/211 [00:56<00:00,  3.75it/s]


Epoch [19/20] | Train Loss: 0.2527, Val Loss: 0.2915 | Tuned Thresold: 0.8433 | Val Accuracy: 0.5149, Precision: 0.5115, Recall: 0.9930, F1: 0.6752


100%|██████████| 211/211 [00:55<00:00,  3.81it/s]


Epoch [20/20] | Train Loss: 0.2491, Val Loss: 0.2948 | Tuned Thresold: 0.7491 | Val Accuracy: 0.5196, Precision: 0.5140, Recall: 0.9859, F1: 0.6758
Training complete. Stats saved.


# Test

In [13]:
# Create model with different backbones and load weights from the best model
backbone = 'resnet18' # 'efficientnet_b0', 'efficientnet_b2', 'resnet18', 'resnet50'
model = SiameseNetwork(model_name=backbone, pretrained=False)
model.load_state_dict(torch.load(f'weights/best_{backbone}_model.pth', weights_only=True))
model.to(device)
criterion = ContrastiveLoss(margin=1.0)

# Test the model
test_stats, cm = test(model, test_loader, criterion=criterion, device=device, threshold=best_threshold)
performance_data = {
    "test_stats": test_stats,
    "confusion_matrix": cm.tolist() if isinstance(cm, np.ndarray) else cm
}
save_stats_path = f'stats/test/performance_{backbone}_stats.json'
os.makedirs(os.path.dirname(save_stats_path), exist_ok=True)

with open(save_stats_path, "w") as f:
    json.dump(performance_data, f, indent=4)
print("Testing complete. Stats saved.")

Test Loss: 0.4447, Test Accuracy: 0.5071, Test Precision: 0.5071, Test Recall: 1.0000, Test F1: 0.6730
Testing complete. Stats saved.
